# 01 — Pozos: perfilado del maestro de pozos (Capítulo IV)

**Objetivo:** conocer la estructura y la calidad del archivo **antes de limpiar nada**.
Este notebook solo *mide*: no modifica ni guarda datos.

| Dato | Valor |
| --- | --- |
| Archivo de origen | `data/raw/capitulo-iv-pozos.csv` |
| Fuente | Secretaría de Energía — ver `data/raw/README.md` |
| Qué contiene | Datos maestros de cada pozo: empresa, yacimiento, cuenca, provincia, profundidad, estado |
| Política aplicada | `POLITICA_CALIDAD_DATOS.md` — paso 1: **Perfilar** |
| Etapa siguiente | `02_pozos_limpieza.ipynb` — aplica las decisiones sobre estos hallazgos |

Las dimensiones de calidad que se revisan son: **completitud, unicidad, validez y consistencia**.

**Trazabilidad:** cada problema detectado recibe un código **H-xx** (hallazgo) en la tabla de la sección 9.
En `02_pozos_limpieza.ipynb`, cada regla de limpieza **R-xx** indica qué hallazgo resuelve.

## 1. Preparar el entorno

Importamos **pandas**, la librería de Python para trabajar con tablas.
Mostramos su versión porque forma parte de la reproducibilidad: con otra versión, algún resultado podría variar.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 50)
print("Versión de pandas:", pd.__version__)

Versión de pandas: 3.0.3


## 2. Cargar el archivo original

Leemos el CSV desde `data/raw`. Solo se **lee**: el archivo original no se toca.

- La ruta empieza con `../` porque el notebook está en la carpeta `notebooks/` y hay que "subir" un nivel.
- `low_memory=False` hace que pandas lea el archivo completo antes de decidir el tipo de cada columna, así evita tipos mezclados.

In [2]:
RUTA_RAW = "../data/raw/capitulo-iv-pozos.csv"

df = pd.read_csv(RUTA_RAW, low_memory=False)
print(f"Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")

Filas: 85,611  |  Columnas: 26


## 3. Estructura: columnas y tipos de datos

`info()` muestra cada columna, cuántos valores no vacíos tiene y su **tipo de dato**
(`int64` = número entero, `float64` = número con decimales, `str` = texto).

Es importante porque un tipo incorrecto trae problemas después: por ejemplo, las **fechas** vienen como texto
y así no se pueden ordenar ni restar.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 85611 entries, 0 to 85610
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   sigla                    85611 non-null  str    
 1   idpozo                   85611 non-null  int64  
 2   area                     85611 non-null  str    
 3   cod_area                 85611 non-null  str    
 4   empresa                  84647 non-null  str    
 5   yacimiento               85611 non-null  str    
 6   cod_yacimiento           85611 non-null  str    
 7   formacion                82796 non-null  str    
 8   cuenca                   85611 non-null  str    
 9   provincia                85611 non-null  str    
 10  cota                     85611 non-null  float64
 11  profundidad              85611 non-null  float64
 12  clasificacion            85611 non-null  str    
 13  subclasificacion         85611 non-null  str    
 14  tipo_recurso             85611 no

## 4. Primer vistazo a los datos

Miramos las primeras filas para entender qué representa cada columna.
Usamos `.T` (transpuesta) para ver las columnas como filas, porque son 26 y no entran a lo ancho.

In [4]:
df.head(3).T

,0,1,2
sigla,CH.CH.EaLE.x-1,CH.CH.EaLE.x-2,CH.CH.EaLE-3
idpozo,212,213,214
area,ESTANCIA LA ESCONDIDA,ESTANCIA LA ESCONDIDA,ESTANCIA LA ESCONDIDA
cod_area,ECH,ECH,ECH
empresa,COLHUE HUAPI S.A.,COLHUE HUAPI S.A.,COLHUE HUAPI S.A.
yacimiento,ESTANCIA LA ESCONDIDA,ESTANCIA LA ESCONDIDA,ESTANCIA LA ESCONDIDA
cod_yacimiento,ELA,ELA,ELA
formacion,comodoro rivadavia,comodoro rivadavia,comodoro rivadavia
cuenca,GOLFO SAN JORGE,GOLFO SAN JORGE,GOLFO SAN JORGE
provincia,Chubut,Chubut,Chubut


## 5. Completitud: ¿falta algún dato?

Contamos los **nulos** (celdas vacías) por columna y su porcentaje sobre el total.

Pero un dato puede faltar aunque la celda no esté vacía: esta fuente usa el texto **"No informado"**.
Son **nulos ocultos** y hay que contarlos aparte, porque `isna()` no los detecta.

In [5]:
completitud = pd.DataFrame({
    "nulos": df.isna().sum(),
    "no_informado": (df == "No informado").sum(),
})
completitud["faltantes_total"] = completitud["nulos"] + completitud["no_informado"]
completitud["pct_faltante"] = (completitud["faltantes_total"] / len(df) * 100).round(1)

completitud[completitud["faltantes_total"] > 0].sort_values("pct_faltante", ascending=False)

,nulos,no_informado,faltantes_total,pct_faltante
sub_tipo_recurso,0,80608,80608,94.2
adjiv_fecha_fin_term,36486,0,36486,42.6
adjiv_fecha_inicio_term,36487,0,36487,42.6
adjiv_fecha_fin_perf,34149,0,34149,39.9
adjiv_fecha_inicio_perf,34004,0,34004,39.7
tipo_recurso,0,21859,21859,25.5
subclasificacion,0,18199,18199,21.3
clasificacion,0,18199,18199,21.3
formacion,2815,0,2815,3.3
empresa,964,0,964,1.1


## 6. Unicidad: ¿hay registros repetidos?

Revisamos tres niveles:

1. **Filas completas duplicadas**: la misma fila exacta dos veces.
2. **`idpozo` repetido**: es la clave del maestro; si se repite, los cruces con los archivos de producción darían resultados duplicados.
3. **`sigla` repetida**: la sigla es el nombre del pozo. Si se repite con distinto `idpozo`, hay que entender por qué antes de decidir nada.

In [6]:
print("Filas completas duplicadas:", df.duplicated().sum())
print("idpozo repetidos:          ", df["idpozo"].duplicated().sum())

filas_sigla_repetida = df["sigla"].duplicated(keep=False)
print("Filas con sigla repetida:  ", filas_sigla_repetida.sum(),
      f"({filas_sigla_repetida.mean() * 100:.1f}% del total)")

Filas completas duplicadas: 0
idpozo repetidos:           0
Filas con sigla repetida:   13421 (15.7% del total)


Veamos algunos ejemplos de siglas repetidas para entender el caso.

In [7]:
df[filas_sigla_repetida].sort_values("sigla")[
    ["sigla", "idpozo", "empresa", "provincia", "tipoestado"]
].head(8)

,sigla,idpozo,empresa,provincia,tipoestado
83279,ACO.RN.TA-4001,164626,TANGO ENERGY ARGENTINA S.A.,Rio Negro,Extracción Efectiva
83280,ACO.RN.TA-4001,164627,TANGO ENERGY ARGENTINA S.A.,Rio Negro,Parado Transitoriamente
83326,ACO.RN.TA-4002,164673,TANGO ENERGY ARGENTINA S.A.,Rio Negro,Parado Transitoriamente
83327,ACO.RN.TA-4002,164674,TANGO ENERGY ARGENTINA S.A.,Rio Negro,Extracción Efectiva
83328,ACO.RN.TA-4002,164675,TANGO ENERGY ARGENTINA S.A.,Rio Negro,Parado Transitoriamente
63695,AEA.NQ.RCo.x-2001,133430,OILSTONE ENERGIA S.A.,Neuquén,Extracción Efectiva
81396,AEA.NQ.RCo.x-2001,162701,OILSTONE ENERGIA S.A.,Neuquén,Extracción Efectiva
76528,AEC.Nq.AC.a-2002,157635,OILSTONE ENERGIA S.A.,Neuquén,Extracción Efectiva


También revisamos si hay pozos distintos con **exactamente la misma ubicación** (columna `geojson`, que tiene las coordenadas).
Puede ser legítimo (por ejemplo, varios pozos desde una misma plataforma) o un error de carga.

In [8]:
filas_ubicacion_repetida = df["geojson"].duplicated(keep=False)
print("Filas que comparten ubicación con otro pozo:", filas_ubicacion_repetida.sum(),
      f"({filas_ubicacion_repetida.mean() * 100:.1f}%)")

Filas que comparten ubicación con otro pozo: 12104 (14.1%)


## 7. Validez: ¿los números tienen sentido?

`describe()` resume las columnas numéricas: mínimo, máximo, promedio y cuartiles.
Buscamos valores **imposibles o sospechosos**:

- **`profundidad`** (metros): un pozo no puede medir 0 m, y el más profundo del mundo ronda los 12.000 m.
- **`cota`** (metros sobre el nivel del mar): valores negativos o de 0 m en zonas de montaña son raros.

In [9]:
df[["cota", "profundidad"]].describe().round(1)

,cota,profundidad
count,85611.0,85611.0
mean,499.2,1717.8
std,308.5,1749.1
min,-100.0,0.0
25%,307.9,1032.0
50%,428.7,1595.0
75%,644.4,2352.0
max,5543.4,378939.0


Contamos los casos sospechosos y miramos los pozos más profundos.

In [10]:
print("Profundidad = 0:        ", (df["profundidad"] == 0).sum())
print("Profundidad > 10.000 m: ", (df["profundidad"] > 10_000).sum())
print("Cota negativa:          ", (df["cota"] < 0).sum())
print("Cota = 0:               ", (df["cota"] == 0).sum())

df.nlargest(7, "profundidad")[["idpozo", "sigla", "provincia", "profundidad"]]

Profundidad = 0:         10299
Profundidad > 10.000 m:  7
Cota negativa:           1
Cota = 0:                225


,idpozo,sigla,provincia,profundidad
75735,156804,YEA.RN.EFO-152(d),Rio Negro,378939.0
41544,109212,YPF.Ch.E.a-669,Chubut,27064.0
21214,69901,YPF.Md.PC-33,Mendoza,24223.0
34636,102326,YPF.SC.CG-441,Santa Cruz,20680.0
37495,105170,YPF.SC.CS-590,Santa Cruz,15273.0
4722,10269,TA.CMA-1.CullN-1,Tierra del Fuego,11021.0
29364,97069,YPF.SC.LP.a-2162,Santa Cruz,10600.0


### Validez de las fechas

Las cuatro columnas `adjiv_fecha_*` registran cuándo empezó y terminó la perforación y la terminación del pozo.
Vienen como **texto**, así que las convertimos a fecha **en una variable aparte**, sin tocar `df`, solo para medir:

- ¿Hay textos que no se pueden convertir a fecha?
- ¿Aparece la fecha **1900-01-01**? Suele usarse como "fecha de relleno" cuando no se conoce la real.
- ¿Hay fechas de **fin anteriores al inicio**?

In [11]:
columnas_fecha = [
    "adjiv_fecha_inicio_perf", "adjiv_fecha_fin_perf",
    "adjiv_fecha_inicio_term", "adjiv_fecha_fin_term",
]
fechas = df[columnas_fecha].apply(pd.to_datetime, errors="coerce")

resumen_fechas = pd.DataFrame({
    "vacias": df[columnas_fecha].isna().sum(),
    "no_convertibles": (fechas.isna() & df[columnas_fecha].notna()).sum(),
    "igual_1900_01_01": (fechas == "1900-01-01").sum(),
    "minima": fechas.min(),
    "maxima": fechas.max(),
})
resumen_fechas

,vacias,no_convertibles,igual_1900_01_01,minima,maxima
adjiv_fecha_inicio_perf,34004,0,37,1900-01-01,2026-05-06
adjiv_fecha_fin_perf,34149,0,33,1900-01-01,2026-05-22
adjiv_fecha_inicio_term,36487,0,33,1900-01-01,2026-06-01
adjiv_fecha_fin_term,36486,0,33,1900-01-01,2026-06-12


In [12]:
print("Fin de perforación antes del inicio:",
      (fechas["adjiv_fecha_fin_perf"] < fechas["adjiv_fecha_inicio_perf"]).sum())
print("Fin de terminación antes del inicio:",
      (fechas["adjiv_fecha_fin_term"] < fechas["adjiv_fecha_inicio_term"]).sum())

Fin de perforación antes del inicio: 3
Fin de terminación antes del inicio: 0


## 8. Consistencia: ¿los valores se escriben siempre igual?

Para las columnas de **categorías** (valores que se repiten de una lista) miramos cuántos valores distintos hay y cuáles son.
Así aparecen variantes de escritura, mayúsculas/minúsculas mezcladas o categorías raras.

In [13]:
columnas_categoria = [
    "provincia", "cuenca", "clasificacion", "tipo_recurso",
    "sub_tipo_recurso", "tipopozo", "tipoestado", "gasplus",
]
for col in columnas_categoria:
    print(f"--- {col} ({df[col].nunique()} valores distintos)")
    print(df[col].value_counts(dropna=False).to_string(), "\n")

--- provincia (13 valores distintos)
provincia
Santa Cruz          23707
Chubut              22606
Neuquén             19497
Mendoza              8897
Rio Negro            5773
La Pampa             2726
Tierra del Fuego     1242
Salta                 965
Estado Nacional        75
Formosa                65
Jujuy                  46
San Juan                6
Córdoba                 6 

--- cuenca (10 valores distintos)
cuenca
GOLFO SAN JORGE    44390
NEUQUINA           33155
CUYANA              3741
AUSTRAL             3236
NOROESTE            1065
NORESTE               16
LOS BOLSONES           3
CAÑADON ASFALTO        2
ÑIRIHUAU               2
GENERAL LEVALLE        1 

--- clasificacion (5 valores distintos)
clasificacion
EXPLOTACION       57412
No informado      18199
EXPLORACION        5950
SERVICIO           4017
ALMACENAMIENTO       33 

--- tipo_recurso (5 valores distintos)
tipo_recurso
CONVENCIONAL       58320
No informado       21859
NO CONVENCIONAL     5006
SIN RESERVORIO   

### Nombres y códigos

Cada `area` debería tener un único `cod_area`, y cada `yacimiento` un único `cod_yacimiento`.
Si la cantidad de nombres y de códigos no coincide, algún nombre tiene más de un código (o al revés).
Revisamos también si los textos mezclan mayúsculas y minúsculas.

In [14]:
print("Áreas distintas:      ", df["area"].nunique(), " | códigos de área:      ", df["cod_area"].nunique())
print("Yacimientos distintos:", df["yacimiento"].nunique(), " | códigos de yacimiento:", df["cod_yacimiento"].nunique())

areas_con_varios_codigos = df.groupby("area")["cod_area"].nunique()
print("Áreas con más de un código:", (areas_con_varios_codigos > 1).sum())

for col in ["empresa", "formacion", "area", "yacimiento"]:
    valores = df[col].dropna()
    print(f"{col:11} -> todo en mayúsculas: {valores.str.isupper().mean() * 100:5.1f}%"
          f" | todo en minúsculas: {valores.str.islower().mean() * 100:5.1f}%")

Áreas distintas:       457  | códigos de área:       464
Yacimientos distintos: 1184  | códigos de yacimiento: 1316
Áreas con más de un código: 7


empresa     -> todo en mayúsculas:  96.5% | todo en minúsculas:   0.0%
formacion   -> todo en mayúsculas:   0.0% | todo en minúsculas:  99.9%
area        -> todo en mayúsculas:  99.9% | todo en minúsculas:   0.0%
yacimiento  -> todo en mayúsculas:  99.9% | todo en minúsculas:   0.0%


## 9. Registro de hallazgos

Cada problema detectado recibe un **código H-xx** y se mide en filas y porcentaje, según las dimensiones de la política de calidad.
Este registro es el punto de partida de `02_pozos_limpieza.ipynb`: cada regla de limpieza (R-xx) va a indicar qué hallazgo resuelve.

Se calcula con código para que, si el archivo raw se actualiza, el registro se actualice solo.
El **estado** de todos es *Pendiente*: en este notebook solo se diagnostica.

In [15]:
total = len(df)
codigos_repetidos = areas_con_varios_codigos[areas_con_varios_codigos > 1].index

hallazgos = [
    ("Completitud", "empresa vacía", df["empresa"].isna().sum()),
    ("Completitud", "formacion vacía", df["formacion"].isna().sum()),
    ("Completitud", "clasificacion y subclasificacion = 'No informado'", (df["clasificacion"] == "No informado").sum()),
    ("Completitud", "tipo_recurso = 'No informado'", (df["tipo_recurso"] == "No informado").sum()),
    ("Completitud", "sub_tipo_recurso = 'No informado'", (df["sub_tipo_recurso"] == "No informado").sum()),
    ("Completitud", "tipopozo, tipoextraccion y tipoestado = 'No informado'", (df["tipopozo"] == "No informado").sum()),
    ("Completitud", "fecha de inicio de perforación vacía", df["adjiv_fecha_inicio_perf"].isna().sum()),
    ("Unicidad", "sigla repetida con distinto idpozo", filas_sigla_repetida.sum()),
    ("Unicidad", "ubicación compartida con otro pozo", filas_ubicacion_repetida.sum()),
    ("Validez", "profundidad = 0", (df["profundidad"] == 0).sum()),
    ("Validez", "profundidad > 10.000 m", (df["profundidad"] > 10_000).sum()),
    ("Validez", "cota = 0 o negativa", (df["cota"] <= 0).sum()),
    ("Validez", "fecha 1900-01-01 (inicio de perforación)", (fechas["adjiv_fecha_inicio_perf"] == "1900-01-01").sum()),
    ("Validez", "fin de perforación antes del inicio",
     (fechas["adjiv_fecha_fin_perf"] < fechas["adjiv_fecha_inicio_perf"]).sum()),
    ("Validez", "fechas guardadas como texto (4 columnas)", total),
    ("Consistencia", "formacion en minúsculas (el resto de los textos en mayúsculas)", df["formacion"].str.islower().sum()),
    ("Consistencia", "pozos en áreas con más de un código de área", df["area"].isin(codigos_repetidos).sum()),
    ("Consistencia", "provincia = 'Estado Nacional' (no es una provincia)", (df["provincia"] == "Estado Nacional").sum()),
]

registro_hallazgos = pd.DataFrame(hallazgos, columns=["dimension", "hallazgo", "filas"])
registro_hallazgos.insert(0, "id", [f"H-{i:02d}" for i in range(1, len(hallazgos) + 1)])
registro_hallazgos["pct"] = (registro_hallazgos["filas"] / total * 100).round(2)
registro_hallazgos["estado"] = "Pendiente"
registro_hallazgos

,id,dimension,hallazgo,filas,pct,estado
0,H-01,Completitud,empresa vacía,964,1.13,Pendiente
1,H-02,Completitud,formacion vacía,2815,3.29,Pendiente
2,H-03,Completitud,clasificacion y subclasificacion = 'No informado',18199,21.26,Pendiente
3,H-04,Completitud,tipo_recurso = 'No informado',21859,25.53,Pendiente
4,H-05,Completitud,sub_tipo_recurso = 'No informado',80608,94.16,Pendiente
5,H-06,Completitud,"tipopozo, tipoextraccion y tipoestado = 'No in...",55,0.06,Pendiente
6,H-07,Completitud,fecha de inicio de perforación vacía,34004,39.72,Pendiente
7,H-08,Unicidad,sigla repetida con distinto idpozo,13421,15.68,Pendiente
8,H-09,Unicidad,ubicación compartida con otro pozo,12104,14.14,Pendiente
9,H-10,Validez,profundidad = 0,10299,12.03,Pendiente


## 10. Conclusiones y decisiones pendientes

**Lo que está bien**
- No hay filas duplicadas ni `idpozo` repetidos: `idpozo` sirve como **clave única** para cruzar con los archivos de producción.
- Las fechas que tienen valor se pueden convertir todas a fecha.

**Lo que hay que decidir en `02_pozos_limpieza.ipynb`** (cada decisión, con sus opciones, pros y contras, la toma la responsable del proyecto):

| Decisión | Hallazgos | Pregunta |
| --- | --- | --- |
| D-1 Nulos ocultos | H-03, H-04, H-05, H-06 | ¿Convertir "No informado" en nulo real, o dejarlo como categoría? |
| D-2 Datos faltantes | H-01, H-02, H-07 | ¿Marcar, completar o dejar vacíos? |
| D-3 Siglas repetidas | H-08 | ¿Son re-ingresos o pozos rama del mismo pozo? Investigar antes de decidir. |
| D-4 Ubicaciones compartidas | H-09 | ¿Plataformas con varios pozos o error de carga? |
| D-5 Profundidad | H-10, H-11 | ¿Marcar como sospechosos, corregir o dejar? El valor de 378.939 m es imposible. |
| D-6 Cota | H-12 | ¿Cota 0 o negativa es un dato real o un valor de relleno? |
| D-7 Fechas | H-13, H-14, H-15 | Convertir a tipo fecha; ¿qué hacer con 1900-01-01 y con los fines anteriores al inicio? |
| D-8 Textos | H-16 | ¿Unificar mayúsculas/minúsculas? |
| D-9 Códigos | H-17 | ¿Cuál es la referencia: el nombre del área o su código? |
| D-10 Jurisdicción | H-18 | "Estado Nacional" probablemente son pozos costa afuera (offshore). ¿Se deja o se recategoriza? |

**Otra observación (no es un problema de calidad):** las columnas `geojson` y `geom` guardan la misma ubicación en dos formatos.
Evaluar en la limpieza si se conservan las dos.

> Criterio de la política: *ante la duda, marcar antes que eliminar*.
> `POLITICA_CALIDAD_DATOS.md` es la referencia para el diagnóstico y no se modifica;
> las reglas de limpieza (R-xx) se registran en `02_pozos_limpieza.ipynb`, cada una con el hallazgo (H-xx) que resuelve.